# NeoNatal Watch AI — Phase 5: XGBoost Baseline

> **DEMO / SYNTHETIC DATA — NOT FOR CLINICAL USE**
>
> This notebook walks through training our first Machine Learning model (XGBoost) using the temporal features engineered in Phase 4.

---

## Why XGBoost?
- **Interpretable**: It's easy to see *which* vital signs and rolling features drive the predictions.
- **Handles Imbalance**: Using `scale_pos_weight`, we can force the model to pay extra attention to the rare deterioration events (which only occur ~0.3% of the time).
- **Robust**: Excellent baseline to compare against Deep Learning models in Phase 6.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from ml.evaluation.metrics import evaluate_model

print('Imports OK')

In [ ]:
# Load feature-engineered data
train_df = pd.read_csv('../data/processed/train_features.csv')
val_df   = pd.read_csv('../data/processed/val_features.csv')
test_df  = pd.read_csv('../data/processed/test_features.csv')

exclude = ['timestamp', 'patient_id', 'deterioration_label', 'clinical_event', 'data_source', 'data_warning']
feature_cols = [c for c in train_df.columns if c not in exclude]

X_train, y_train = train_df[feature_cols], train_df['deterioration_label']
X_val, y_val     = val_df[feature_cols], val_df['deterioration_label']
X_test, y_test   = test_df[feature_cols], test_df['deterioration_label']

print(f"Training samples: {len(X_train)} | Features: {len(feature_cols)}")

In [ ]:
# Handling extreme class imbalance
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = min(n_neg / n_pos, 50.0) if n_pos > 0 else 1.0

print(f"Negatives: {n_neg}, Positives: {n_pos}")
print(f"Setting scale_pos_weight to {scale_pos_weight:.2f}")

In [ ]:
# Initialize and Train Model
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    early_stopping_rounds=20,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

In [ ]:
# Evaluate Model
y_test_probs = model.predict_proba(X_test)[:, 1]

metrics = evaluate_model(
    y_true=y_test,
    y_probs=y_test_probs,
    model_name="XGBoost_Notebook",
    save_dir="../reports/figures"
)

In [ ]:
# Feature Importance
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.barh(importance['Feature'][::-1], importance['Importance'][::-1], color='teal')
plt.title('Top 15 Most Important Features for Predicting Deterioration')
plt.xlabel('Importance Score')
plt.show()